# Verify landmark patch extraction

**Part 1 (landmark overlay check) can be run now. Part 2 (patch shapes) cannot be run until `landmark-pretrain`'s patch
extraction exists** (LRLP selection, 32x32 patch cropping, nose-tip
alignment). It is a placeholder, checked in now so the verification
contract is written down before that code exists, not reverse-engineered
after.

**What "looks right" means, per Sheng et al. 2022's exact spec:**
- Patch tensor shape: `(K=38, T, 32, 32)` -- K selected landmarks, T
  frames, 32x32 grayscale pixel patches.
- Coordinate tensor shape: `(K=38, 2, T)` -- K landmarks, (x, y), T
  frames, nose-tip-relative.
- `K == 38` specifically requires the exact LRLP index list to have been
  resolved first (CLAUDE.md flags this as still unresolved -- do not
  guess it; a wrong index selection would silently corrupt the local
  stream with no error thrown).

# Part 1: overlay GRID landmarks on real frames

Randomly (uniform, seed 42) pick 5 GRID clips from the manifest saved by
notebook 01 (`grid_manifest.csv`, in this same folder), pick 2 random frames
from each, and plot the landmarks on top of the frame.

**Frame/landmark alignment.** The landmark files were produced by decoding the
video with TorchCodec frame by frame and running the detector on each frame,
so `landmarks[i]` belongs to decoded frame `i`. The "at most 3 frames"
difference seen in `check_frame_count_vs_duration` compares the landmark count
against the AUDIO duration x 25fps, not against the decoded video frame count.
So the first thing this notebook checks is `len(landmarks) == number of decoded
frames`. If they are equal, indexing is exact and there is no ambiguity. If
they differ, any drift can only build up towards the end of the clip, so frames
are drawn from the first half of the clip, where the index is still reliable.
Frames whose landmark is `None` (no face detected) are skipped.

**What "looks right" means:** the 68 points sit on the face (jaw, brows, nose,
eyes, mouth) and the mouth points (orange) follow the lips in both frames.

In [ ]:
import random
import pickle
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from torchcodec.decoders import VideoDecoder

SEED = 42
NUM_CLIPS = 5
FRAMES_PER_CLIP = 2
MANIFEST_DIR = Path.cwd()  # notebook 01 saves its CSVs in this same folder

grid_manifest = pd.read_csv(MANIFEST_DIR / "grid_manifest.csv")
print(grid_manifest.shape)

In [ ]:
rng = random.Random(SEED)
sampled_rows = grid_manifest.sample(n=NUM_CLIPS, random_state=SEED)  # uniform, without replacement

samples = []  # (sample_id, frame_index, frame_rgb, landmarks_68x2)
for _, row in sampled_rows.iterrows():
    decoder = VideoDecoder(row["video_path"], dimension_order="NHWC")
    with open(row["landmark_path"], "rb") as f:
        landmarks = pickle.load(f)

    num_video_frames, num_landmarks = len(decoder), len(landmarks)
    print(f"{row['sample_id']}: video frames={num_video_frames}, landmark frames={num_landmarks}")

    # Indexing is exact when the counts match; otherwise stay in the first half.
    usable = num_video_frames if num_video_frames == num_landmarks else min(num_video_frames, num_landmarks) // 2
    candidates = [i for i in range(usable) if landmarks[i] is not None]
    for frame_index in rng.sample(candidates, k=FRAMES_PER_CLIP):
        samples.append((row["sample_id"], frame_index, decoder[frame_index].numpy(), landmarks[frame_index]))

In [ ]:
MOUTH = slice(48, 68)  # iBUG 68-point layout: mouth = 48-67

fig, axes = plt.subplots(NUM_CLIPS, FRAMES_PER_CLIP, figsize=(4 * FRAMES_PER_CLIP, 3.5 * NUM_CLIPS))
for ax, (sample_id, frame_index, frame, pts) in zip(axes.ravel(), samples):
    ax.imshow(frame)
    mouth_mask = np.zeros(len(pts), dtype=bool)
    mouth_mask[MOUTH] = True
    ax.scatter(pts[~mouth_mask, 0], pts[~mouth_mask, 1], s=8, c="lime")   # 48 non-mouth points
    ax.scatter(pts[mouth_mask, 0], pts[mouth_mask, 1], s=8, c="orange")   # 20 mouth points
    ax.set_title(f"{sample_id}  frame {frame_index}  ({len(pts)} pts)", fontsize=9)
    ax.axis("off")
plt.tight_layout()
plt.show()

## Part 1 checklist

- [ ] `video frames == landmark frames` for the sampled clips (if not, note how
      many differ and confirm the plotted first-half frames still look aligned).
- [ ] Landmarks sit on the face in all 10 plots, mouth points follow the lips.
- [ ] Any clip with clearly shifted or missing landmarks is noted and its
      `sample_id` inspected.

---

# Part 2: landmark patch extraction (waiting for landmark-pretrain)

In [ ]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import pickle

import torchvision

# TODO: import once landmark-pretrain's patch extraction module exists, e.g.:
# from fusion_avsr.models.landmark import extract_patches

## Config

In [ ]:
# TODO: fill in a real clip's video + landmark paths from notebook 01's manifest
SAMPLE_VIDEO_PATH = Path("REPLACE_ME.mp4")
SAMPLE_LANDMARK_PATH = Path("REPLACE_ME.pkl")

## Load one real clip's video frames + landmark coordinates

In [ ]:
video_frames, _audio, _info = torchvision.io.read_video(str(SAMPLE_VIDEO_PATH), pts_unit="sec")

with open(SAMPLE_LANDMARK_PATH, "rb") as f:
    landmarks = pickle.load(f)  # list of (68, 2) arrays, one per frame

print(f"video frames: {video_frames.shape[0]}, landmark frames: {len(landmarks)}")

## Extract patches + aligned coordinates (once implemented)

In [ ]:
# TODO once landmark-pretrain's patch extraction exists:
# patch_tensor, coord_tensor = extract_patches(video_frames, landmarks)
# print(f"patch_tensor shape: {tuple(patch_tensor.shape)}")   # expect (38, T, 32, 32)
# print(f"coord_tensor shape: {tuple(coord_tensor.shape)}")   # expect (38, 2, T)

## TODO checklist (once the above is filled in)

- [ ] `patch_tensor.shape == (38, T, 32, 32)`, `dtype` grayscale-appropriate
      (e.g. `uint8` or normalized `float32`, matching the LMFE 3D CNN's
      expected input).
- [ ] `coord_tensor.shape == (38, 2, T)`, coordinates nose-tip-relative
      (not raw pixel coordinates).
- [ ] `T` matches the clip's actual frame count (same as
      `len(landmarks)` above).
- [ ] The 38 selected landmark indices are the ones resolved from a
      released reference implementation or careful reasoning from the
      68-point layout -- NOT guessed. Cross-reference against
      CLAUDE.md's "Still unresolved" note before trusting this.